In [1]:
print("Installing required packages...")
!pip install -q opencv-python-headless numpy ultralytics torch torchvision mediapipe scikit-learn pandas
print("✓ All packages installed!")


Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 17.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
✓ All packages installed!


In [2]:
import cv2
import numpy as np
from collections import deque
import pandas as pd
from datetime import timedelta
from google.colab import files
import os

try:
    from ultralytics import YOLO
    import torch
    import mediapipe as mp
    print("✓ All libraries imported successfully!")
except ImportError as e:
    print(f"Error importing: {e}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ All libraries imported successfully!


In [3]:
print("Please upload your badminton video file:")
uploaded = files.upload()

# Get the uploaded video filename
video_filename = list(uploaded.keys())[0]
print(f"✓ Video uploaded: {video_filename}")

Please upload your badminton video file:


Saving Sample.mp4 to Sample.mp4
✓ Video uploaded: Sample.mp4


In [4]:
class BadmintonAnalyzer:
    def __init__(self, video_path, player_a_name="An Se Young", player_b_name="Ratchanok Intanon"):
        self.video_path = video_path
        self.cap = cv2.VideoCapture(video_path)
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        self.width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        self.player_a_name = player_a_name
        self.player_b_name = player_b_name

        print("Loading AI models (this may take a minute)...")
        self.player_detector = YOLO('yolov8n-pose.pt')
        self.object_detector = YOLO('yolov8n.pt')

        self.pose = mp.solutions.pose.Pose(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

        self.shuttlecock_tracker = ShuttlecockTracker()
        self.court_detector = CourtDetector()
        self.shot_classifier = ShotClassifier()
        self.rally_detector = RallyDetector(self.fps)

        self.score_a = 0
        self.score_b = 0
        self.rallies = []

        print(f"✓ Initialized: {self.fps} FPS, {self.width}x{self.height}")
        print(f"✓ Players: {self.player_a_name} vs {self.player_b_name}")

    def analyze_video(self):
        """Main analysis pipeline"""
        frame_count = 0
        current_rally = None
        total_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))

        print("\nDetecting badminton court...")
        ret, first_frame = self.cap.read()
        if ret:
            self.court_detector.detect_court(first_frame)
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

        print("Starting video analysis...")
        print("This may take several minutes depending on video length...")

        while self.cap.isOpened():
            ret, frame = self.cap.read()
            if not ret:
                break

            current_time = frame_count / self.fps

            players = self.detect_players_with_pose(frame)
            shuttlecock = self.shuttlecock_tracker.track(frame)
            rally_state = self.rally_detector.update(frame, shuttlecock, players, current_time)

            if rally_state == "START":
                current_rally = Rally(current_time, self.court_detector,
                                     self.player_a_name, self.player_b_name)

            elif rally_state == "ACTIVE" and current_rally:
                current_rally.add_frame_data(frame, shuttlecock, players, current_time)

            elif rally_state == "END" and current_rally:
                current_rally.end_time = current_time
                rally_result = self.analyze_rally(current_rally, frame)
                self.rallies.append(rally_result)

                print(f"✓ Rally {len(self.rallies)}: {rally_result['start_time']} - {rally_result['end_time']} | Winner: {rally_result['win_point_player']} | Score: {rally_result['roundscore_A']}-{rally_result['roundscore_B']}")

                current_rally = None

            frame_count += 1

            # Progress indicator
            if frame_count % 600 == 0:
                progress = (frame_count / total_frames) * 100
                print(f"Progress: {progress:.1f}% ({self.format_time(current_time)})")

        self.cap.release()
        print(f"\n✓ Analysis complete! Found {len(self.rallies)} rallies")
        return self.rallies

    def detect_players_with_pose(self, frame):
        """Detect players with pose estimation"""
        results = self.player_detector(frame, verbose=False)

        players = []
        for result in results:
            if result.keypoints is not None:
                for keypoint in result.keypoints:
                    kp = keypoint.xy[0].cpu().numpy()
                    conf = keypoint.conf[0].cpu().numpy()

                    valid_points = kp[conf > 0.5]
                    if len(valid_points) > 0:
                        x_coords = valid_points[:, 0]
                        y_coords = valid_points[:, 1]

                        bbox = [
                            int(x_coords.min()), int(y_coords.min()),
                            int(x_coords.max()), int(y_coords.max())
                        ]

                        pose_features = self.extract_pose_features(kp, conf)

                        players.append({
                            'bbox': bbox,
                            'keypoints': kp,
                            'confidence': conf,
                            'pose_features': pose_features,
                            'center_x': (bbox[0] + bbox[2]) / 2,
                            'center_y': (bbox[1] + bbox[3]) / 2
                        })

        players.sort(key=lambda p: p['center_x'])
        return players[:2]

    def extract_pose_features(self, keypoints, confidence):
        """Extract features from pose keypoints"""
        features = {
            'arm_raised': False,
            'arm_extended': False,
            'body_leaning': 'neutral',
            'stance': 'neutral'
        }

        if len(keypoints) > 16 and len(confidence) > 16:
            if confidence[15] > 0.5 and confidence[11] > 0.5:
                if keypoints[15][1] < keypoints[11][1]:
                    features['arm_raised'] = True

            if confidence[15] > 0.5 and confidence[13] > 0.5:
                wrist = keypoints[15]
                elbow = keypoints[13]
                distance = np.linalg.norm(wrist - elbow)
                if distance > 50:
                    features['arm_extended'] = True

        return features

    def analyze_rally(self, rally, last_frame):
        """Analyze complete rally and determine outcome"""
        winner, loser = self.determine_rally_winner(rally)
        shot_type = self.shot_classifier.classify_shot(rally)
        win_reason, lose_reason = self.determine_reasons(rally, winner, loser)

        if winner == self.player_a_name:
            self.score_a += 1
        else:
            self.score_b += 1

        result = {
            'start_time': self.format_time(rally.start_time),
            'end_time': self.format_time(rally.end_time),
            'win_point_player': winner,
            'win_reason': win_reason,
            'ball_types': shot_type,
            'lose_reason': lose_reason,
            'roundscore_A': self.score_a,
            'roundscore_B': self.score_b
        }

        return result

    def determine_rally_winner(self, rally):
        """Determine winner based on shuttlecock trajectory and landing"""
        if not rally.shuttlecock_positions:
            return self.player_a_name, self.player_b_name

        last_pos = rally.shuttlecock_positions[-1]
        landing_zone = self.court_detector.get_landing_zone(last_pos)
        last_hitter = rally.get_last_hitter()

        if landing_zone in ['out_of_bounds', 'net', 'below_net']:
            winner = self.player_b_name if last_hitter == self.player_a_name else self.player_a_name
            loser = last_hitter
        else:
            winner = last_hitter
            loser = self.player_b_name if winner == self.player_a_name else self.player_a_name

        return winner, loser

    def determine_reasons(self, rally, winner, loser):
        """Determine win and lose reasons"""
        last_pos = rally.shuttlecock_positions[-1] if rally.shuttlecock_positions else None
        landing_zone = self.court_detector.get_landing_zone(last_pos) if last_pos else 'unknown'

        if landing_zone == 'out_of_bounds':
            win_reason = "opponent goes out of bounds"
            lose_reason = "goes out of bounds"
        elif landing_zone == 'net':
            win_reason = "opponent hits the net"
            lose_reason = "hits the net"
        elif landing_zone == 'below_net':
            win_reason = "opponent fails to clear the net"
            lose_reason = "fails to clear the net"
        elif landing_zone == 'valid':
            win_reason = "wins by landing"
            lose_reason = "opponent wins by landing"
        else:
            win_reason = "opponent error"
            lose_reason = "error"

        return win_reason, lose_reason

    def format_time(self, seconds):
        """Format time as MM:SS or HH:MM:SS"""
        total_seconds = int(seconds)
        minutes = total_seconds // 60
        secs = total_seconds % 60

        if minutes >= 60:
            hours = minutes // 60
            minutes = minutes % 60
            return f"{hours:02d}:{minutes:02d}:{secs:02d}"
        else:
            return f"{minutes:02d}:{secs:02d}"

    def save_results(self, output_path='badminton_analysis.csv'):
        """Save results in exact format"""
        df = pd.DataFrame(self.rallies, columns=[
            'start_time', 'end_time', 'win_point_player', 'win_reason',
            'ball_types', 'lose_reason', 'roundscore_A', 'roundscore_B'
        ])

        df.to_csv(output_path, index=False)
        print(f"\n✓ Results saved to {output_path}")
        print(f"  Format: start_time, end_time, win_point_player, win_reason, ball_types, lose_reason, roundscore_A, roundscore_B")
        print(f"  Total rallies: {len(df)}")
        print(f"  Final score: {self.score_a} - {self.score_b}")

        return output_path


class Rally:
    """Stores data for a single rally"""
    def __init__(self, start_time, court_detector, player_a_name, player_b_name):
        self.start_time = start_time
        self.end_time = None
        self.shuttlecock_positions = []
        self.player_positions = []
        self.shots = []
        self.court_detector = court_detector
        self.last_hitter_side = None
        self.player_a_name = player_a_name
        self.player_b_name = player_b_name

    def add_frame_data(self, frame, shuttlecock, players, current_time):
        """Add frame data to rally"""
        if shuttlecock:
            self.shuttlecock_positions.append(shuttlecock)

        self.player_positions.append(players)

        if shuttlecock and len(players) >= 2:
            shot = self.detect_shot(shuttlecock, players)
            if shot:
                self.shots.append(shot)

    def detect_shot(self, shuttlecock, players):
        """Detect if a shot occurred"""
        for i, player in enumerate(players):
            dist = np.linalg.norm(
                np.array([player['center_x'], player['center_y']]) -
                np.array(shuttlecock)
            )

            if dist < 100:
                if player['pose_features']['arm_raised']:
                    self.last_hitter_side = 'A' if i == 0 else 'B'
                    return {
                        'player': i,
                        'position': shuttlecock,
                        'pose': player['pose_features']
                    }
        return None

    def get_last_hitter(self):
        """Get the last player who hit the shuttlecock"""
        if self.last_hitter_side == 'A':
            return self.player_a_name
        else:
            return self.player_b_name


class ShuttlecockTracker:
    """Track shuttlecock using computer vision"""
    def __init__(self):
        self.prev_positions = deque(maxlen=10)
        self.kalman = cv2.KalmanFilter(4, 2)
        self.kalman.measurementMatrix = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], np.float32)
        self.kalman.transitionMatrix = np.array([[1, 0, 1, 0], [0, 1, 0, 1],
                                                  [0, 0, 1, 0], [0, 0, 0, 1]], np.float32)
        self.lost_frames = 0

    def track(self, frame):
        """Track shuttlecock in frame"""
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        lower_white = np.array([0, 0, 200])
        upper_white = np.array([180, 30, 255])
        mask = cv2.inRange(hsv, lower_white, upper_white)

        kernel = np.ones((3, 3), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            for contour in sorted(contours, key=cv2.contourArea, reverse=True)[:5]:
                area = cv2.contourArea(contour)
                if 5 < area < 500:
                    M = cv2.moments(contour)
                    if M["m00"] > 0:
                        cx = int(M["m10"] / M["m00"])
                        cy = int(M["m01"] / M["m00"])

                        measurement = np.array([[np.float32(cx)], [np.float32(cy)]])
                        self.kalman.correct(measurement)

                        self.lost_frames = 0
                        self.prev_positions.append((cx, cy))
                        return (cx, cy)

        self.lost_frames += 1
        if self.lost_frames < 30 and len(self.prev_positions) > 0:
            prediction = self.kalman.predict()
            return (int(prediction[0]), int(prediction[1]))

        return None


class CourtDetector:
    """Detect and analyze badminton court"""
    def __init__(self):
        self.court_lines = None
        self.net_position = None
        self.bounds = None

    def detect_court(self, frame):
        """Detect court lines and boundaries"""
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 50, 150)

        lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=100, maxLineGap=10)

        if lines is not None:
            self.court_lines = lines

            middle_x = frame.shape[1] // 2
            self.net_position = middle_x

            h, w = frame.shape[:2]
            self.bounds = {
                'left': w * 0.1,
                'right': w * 0.9,
                'top': h * 0.2,
                'bottom': h * 0.8,
                'net': middle_x
            }

    def get_landing_zone(self, position):
        """Determine where shuttlecock landed"""
        if position is None or self.bounds is None:
            return 'unknown'

        x, y = position

        if (x < self.bounds['left'] or x > self.bounds['right'] or
            y < self.bounds['top'] or y > self.bounds['bottom']):
            return 'out_of_bounds'

        if abs(x - self.bounds['net']) < 50 and y > self.bounds['bottom'] * 0.7:
            return 'net'

        if abs(x - self.bounds['net']) < 30:
            return 'below_net'

        return 'valid'


class ShotClassifier:
    """Classify badminton shot types"""

    SHOT_TYPES = [
        'smash', 'clear', 'drop', 'net shot', 'drive', 'lob',
        'push', 'cross-court net shot', 'return net', 'back-court drive',
        'defensive return lob', 'defensive return drive', 'rush'
    ]

    def classify_shot(self, rally):
        """Classify the decisive shot in a rally"""
        if not rally.shots or not rally.shuttlecock_positions:
            return 'push'

        if len(rally.shuttlecock_positions) < 3:
            return 'net shot'

        positions = rally.shuttlecock_positions[-10:]

        y_positions = [p[1] for p in positions]
        x_positions = [p[0] for p in positions]

        y_range = max(y_positions) - min(y_positions)
        y_trend = y_positions[-1] - y_positions[0]
        x_range = max(x_positions) - min(x_positions)

        if y_trend < -100:
            if y_range > 150:
                return 'lob'
            else:
                return 'clear'
        elif y_trend > 100:
            if rally.shots and rally.shots[-1].get('pose', {}).get('arm_raised'):
                return 'smash'
            return 'drop'
        elif abs(x_range) > 200:
            return 'drive'
        elif y_positions[-1] > max(y_positions) * 0.9:
            return 'net shot'
        else:
            return 'push'


class RallyDetector:
    """Detect rally start and end"""
    def __init__(self, fps):
        self.fps = fps
        self.state = "IDLE"
        self.frames_without_motion = 0
        self.frames_with_motion = 0

    def update(self, frame, shuttlecock, players, current_time):
        """Update rally state"""
        if shuttlecock is not None:
            self.frames_with_motion += 1
            self.frames_without_motion = 0

            if self.state == "IDLE" and self.frames_with_motion > 5:
                self.state = "START"
                return "START"
            elif self.state in ["START", "ACTIVE"]:
                self.state = "ACTIVE"
                return "ACTIVE"
        else:
            self.frames_without_motion += 1
            self.frames_with_motion = 0

            if self.state == "ACTIVE" and self.frames_without_motion > self.fps * 1:
                self.state = "IDLE"
                self.frames_without_motion = 0
                return "END"

        return self.state

print("✓ All classes defined successfully!")

✓ All classes defined successfully!


In [5]:
# CUSTOMIZE PLAYER NAMES HERE:
PLAYER_A_NAME = "An Se Young"
PLAYER_B_NAME = "Ratchanok Intanon"

print(f"\n{'='*80}")
print("STARTING BADMINTON VIDEO ANALYSIS")
print(f"{'='*80}")

# Initialize analyzer
analyzer = BadmintonAnalyzer(
    video_filename,
    player_a_name=PLAYER_A_NAME,
    player_b_name=PLAYER_B_NAME
)

# Run analysis
results = analyzer.analyze_video()

# Display first 5 rallies
print(f"\n{'='*80}")
print("SAMPLE RESULTS (First 5 rallies)")
print(f"{'='*80}")

for i, result in enumerate(results[:5]):
    print(f"\nRally {i+1}:")
    print(f"  Time: {result['start_time']} - {result['end_time']}")
    print(f"  Winner: {result['win_point_player']}")
    print(f"  Shot: {result['ball_types']}")
    print(f"  Win reason: {result['win_reason']}")
    print(f"  Lose reason: {result['lose_reason']}")
    print(f"  Score: {result['roundscore_A']} - {result['roundscore_B']}")

# Save results
output_file = analyzer.save_results('badminton_analysis.csv')

print(f"\n{'='*80}")
print("✓ ANALYSIS COMPLETE!")
print(f"✓ Total rallies detected: {len(results)}")
print(f"✓ Final Score: {analyzer.score_a} - {analyzer.score_b}")
print(f"{'='*80}")


STARTING BADMINTON VIDEO ANALYSIS
Loading AI models (this may take a minute)...
✓ Initialized: 30.0 FPS, 1920x1080
✓ Players: An Se Young vs Ratchanok Intanon

Detecting badminton court...
Starting video analysis...
This may take several minutes depending on video length...
Progress: 25.0% (00:19)


/tmp/ipython-input-1037937062.py:332: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return (int(prediction[0]), int(prediction[1]))


Progress: 50.0% (00:39)
✓ Rally 1: 00:37 - 00:40 | Winner: Ratchanok Intanon | Score: 0-1
Progress: 75.0% (00:59)
✓ Rally 2: 01:01 - 01:03 | Winner: Ratchanok Intanon | Score: 0-2
✓ Rally 3: 01:05 - 01:09 | Winner: An Se Young | Score: 1-2
✓ Rally 4: 01:15 - 01:17 | Winner: An Se Young | Score: 2-2
Progress: 100.0% (01:19)

✓ Analysis complete! Found 4 rallies

SAMPLE RESULTS (First 5 rallies)

Rally 1:
  Time: 00:37 - 00:40
  Winner: Ratchanok Intanon
  Shot: push
  Win reason: wins by landing
  Lose reason: opponent wins by landing
  Score: 0 - 1

Rally 2:
  Time: 01:01 - 01:03
  Winner: Ratchanok Intanon
  Shot: push
  Win reason: wins by landing
  Lose reason: opponent wins by landing
  Score: 0 - 2

Rally 3:
  Time: 01:05 - 01:09
  Winner: An Se Young
  Shot: push
  Win reason: opponent goes out of bounds
  Lose reason: goes out of bounds
  Score: 1 - 2

Rally 4:
  Time: 01:15 - 01:17
  Winner: An Se Young
  Shot: push
  Win reason: opponent goes out of bounds
  Lose reason: goes 

In [6]:
# Display CSV preview
print("\nCSV FILE PREVIEW:")
print("="*80)
df = pd.read_csv('badminton_analysis.csv')
print(df.head(10))
print(f"\nTotal rows: {len(df)}")

# Download the CSV file
print("\n📥 Downloading CSV file...")
files.download('badminton_analysis.csv')
print("✓ Download complete! Check your downloads folder.")


CSV FILE PREVIEW:
  start_time end_time   win_point_player                   win_reason  \
0      00:37    00:40  Ratchanok Intanon              wins by landing   
1      01:01    01:03  Ratchanok Intanon              wins by landing   
2      01:05    01:09        An Se Young  opponent goes out of bounds   
3      01:15    01:17        An Se Young  opponent goes out of bounds   

  ball_types               lose_reason  roundscore_A  roundscore_B  
0       push  opponent wins by landing             0             1  
1       push  opponent wins by landing             0             2  
2       push        goes out of bounds             1             2  
3       push        goes out of bounds             2             2  

Total rows: 4

📥 Downloading CSV file...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download complete! Check your downloads folder.
